In [ ]:
# # Week 7 – L2 Experiments Design & Diagnostics (E5–E8)
# 
# This notebook prepares the data and diagnostics for the L2 (MA trend) label
# before running the core L2 experiments E5–E8:
# - E5: Logistic + L2 + F_CORE + V1
# - E6: Linear SVM + L2 + F_CORE + V1
# - E7: Logistic + L2 + F_CORE + V2
# - E8: Linear SVM + L2 + F_CORE + V2
# 

import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
DERIVED_DIR = DATA_DIR / "derived"
PROCESSED_DIR = DATA_DIR / "processed"

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("DERIVED_DIR:", DERIVED_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


In [ ]:
# Load F_CORE features and L2 labels
# 
# We reuse:
# - `data/derived/mru_usd_features_all_week5.csv` for F_CORE
# - `data/derived/mru_usd_labels_L2_ma10_50_h5.csv` for L2
# 
# We will:
# - Parse the `date` column as datetime.
# - Inner-join on `date` to ensure alignment.
# - Detect the L2 label column , preferring an existing `y_L2`
#   column if present .
# - Standardize its name to `y_L2` **inside this notebook only** .

# %%
features_path = DERIVED_DIR / "mru_usd_features_all_week5.csv"
labels_l2_path = DERIVED_DIR / "mru_usd_labels_L2_ma10_50_h5.csv"

print("Features file:", features_path)
print("L2 labels file:", labels_l2_path)

df_feat = pd.read_csv(features_path, parse_dates=["date"])
df_l2 = pd.read_csv(labels_l2_path, parse_dates=["date"])

print("\nFeatures head:")
print(df_feat.head())

print("\nL2 labels head:")
print(df_l2.head())

print("\nShapes: features =", df_feat.shape, ", L2 labels =", df_l2.shape)
print("\nL2 columns:", list(df_l2.columns))


In [ ]:
# 1.1 Detect and standardize the L2 label column
# 

if "y_L2" in df_l2.columns:
    l2_label_col = "y_L2"
else:
    non_date_cols = [c for c in df_l2.columns if c.lower() != "date"]
    if len(non_date_cols) == 1:
        l2_label_col = non_date_cols[0]
    else:
        raise ValueError(
            "Could not uniquely determine L2 label column.\n"
            f"Non-date columns found: {non_date_cols}\n"
            "Please update this cell to point explicitly to the correct label column."
        )

print("Using L2 label column:", l2_label_col)

if l2_label_col != "y_L2":
    df_l2 = df_l2.rename(columns={l2_label_col: "y_L2"})
    print(f"Renamed {l2_label_col} → 'y_L2' inside this notebook.")

df = (
    pd.merge(df_feat, df_l2, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nJoined dataframe head:")
print(df.head())
print("\nJoined dataframe shape:", df.shape)

print("\nNA counts per column (joined df):")
print(df.isna().sum().sort_values(ascending=False))

In [ ]:
#  1.2 Global L2 label distribution (pre-split)

global_l2_dist = df["y_L2"].value_counts().sort_index()
global_l2_prop = df["y_L2"].value_counts(normalize=True).sort_index()

global_l2_summary = pd.DataFrame(
    {"count": global_l2_dist, "proportion": global_l2_prop}
)

print(global_l2_summary)


In [ ]:
# Define V1 and V2 split logic

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

date_min, date_max = df["date"].min(), df["date"].max()
print("Date range in joined df:", date_min, "→", date_max)

In [ ]:
# 2.1 V1 – Single chronological split

V1_TRAIN_END = pd.Timestamp("2015-12-31")
V1_VAL_END = pd.Timestamp("2019-12-31")

mask_v1_train = df["date"] <= V1_TRAIN_END
mask_v1_val = (df["date"] > V1_TRAIN_END) & (df["date"] <= V1_VAL_END)
mask_v1_test = df["date"] > V1_VAL_END

print("V1 counts:")
print("  train:", mask_v1_train.sum())
print("  val  :", mask_v1_val.sum())
print("  test :", mask_v1_test.sum())

print("\nV1 date ranges:")
for name, mask in [("train", mask_v1_train), ("val", mask_v1_val), ("test", mask_v1_test)]:
    if mask.sum() > 0:
        print(f"  {name}: {df.loc[mask, 'date'].iloc[0]} → {df.loc[mask, 'date'].iloc[-1]}")
    else:
        print(f"  {name}: (empty)")

In [ ]:
#  V2 – Walk-forward folds (F1–F4)

V2_FOLDS = {
    "F1": {
        "train_end": pd.Timestamp("2009-12-31"),
        "test_start": pd.Timestamp("2010-01-01"),
        "test_end": pd.Timestamp("2011-12-30"),
    },
    "F2": {
        "train_end": pd.Timestamp("2011-12-30"),
        "test_start": pd.Timestamp("2012-01-02"),
        "test_end": pd.Timestamp("2013-12-31"),
    },
    "F3": {
        "train_end": pd.Timestamp("2013-12-31"),
        "test_start": pd.Timestamp("2014-01-01"),
        "test_end": pd.Timestamp("2015-12-31"),
    },
    "F4": {
        "train_end": pd.Timestamp("2015-12-31"),
        "test_start": pd.Timestamp("2016-01-01"),
        "test_end": pd.Timestamp("2019-12-31"),
    },
}

v2_masks = {}

for fold_id, cfg in V2_FOLDS.items():
    train_end = cfg["train_end"]
    test_start = cfg["test_start"]
    test_end = cfg["test_end"]

    train_mask = df["date"] <= train_end
    test_mask = (df["date"] >= test_start) & (df["date"] <= test_end)

    v2_masks[fold_id] = {"train": train_mask, "test": test_mask}

    print(f"\nFold {fold_id}:")
    print("  train count:", train_mask.sum())
    print("  test  count:", test_mask.sum())

    if train_mask.sum() > 0:
        print("  train dates:", df.loc[train_mask, "date"].iloc[0], "→", df.loc[train_mask, "date"].iloc[-1])
    if test_mask.sum() > 0:
        print("  test  dates:", df.loc[test_mask, "date"].iloc[0], "→", df.loc[test_mask, "date"].iloc[-1])



In [ ]:
#  Helper functions for building X, y and summarizing labels

F_CORE_COLS = [
    "f_r_lag1",
    "f_past_ret_5",
    "f_past_ret_20",
    "f_vol_10",
    "f_vol_30",
]

LABEL_COL_L2 = "y_L2"


def check_feature_columns(df_in, feature_cols):
    missing = [c for c in feature_cols if c not in df_in.columns]
    if missing:
        raise ValueError(f"Missing feature columns in df: {missing}")


def build_xy_v1(df_in, split, feature_cols=F_CORE_COLS, label_col=LABEL_COL_L2):
    """
    Build (df_split, X, y) for V1 split: 'train', 'val', or 'test'.
    """
    if split not in {"train", "val", "test"}:
        raise ValueError(f"Invalid V1 split: {split}")

    if split == "train":
        mask = mask_v1_train
    elif split == "val":
        mask = mask_v1_val
    else:
        mask = mask_v1_test

    check_feature_columns(df_in, feature_cols)

    df_split = df_in.loc[mask].copy()
    X = df_split[feature_cols].values
    y = df_split[label_col].values

    return df_split, X, y


def build_xy_v2_fold(df_in, fold_id, split, feature_cols=F_CORE_COLS, label_col=LABEL_COL_L2):
    """
    Build (df_split, X, y) for V2 fold: F1–F4 and split 'train' or 'test'.
    """
    if fold_id not in v2_masks:
        raise ValueError(f"Unknown fold_id: {fold_id}")
    if split not in {"train", "test"}:
        raise ValueError(f"Invalid V2 split: {split}")

    mask = v2_masks[fold_id][split]
    check_feature_columns(df_in, feature_cols)

    df_split = df_in.loc[mask].copy()
    X = df_split[feature_cols].values
    y = df_split[label_col].values

    return df_split, X, y


def summarize_label_distribution(y, label_name="y"):
    """
    Return a small DataFrame with counts and proportions for a 1D label array.
    NaN labels are ignored (value_counts default behavior).
    """
    s = pd.Series(y, name=label_name)
    counts = s.value_counts().sort_index()
    props = s.value_counts(normalize=True).sort_index()
    out = pd.DataFrame({"count": counts, "proportion": props})
    return out


def run_length_distribution(series):
    """
    Compute run-length distribution for a 1D pandas Series (e.g., y_L2),
    returning a DataFrame with columns: value, run_length, count.

    """
    values = series.values
    if len(values) == 0:
        return pd.DataFrame(columns=["value", "run_length", "count"])

    run_values = []
    run_lengths = []

    current_value = values[0]
    current_len = 1

    for v in values[1:]:
        if v == current_value:
            current_len += 1
        else:
            run_values.append(current_value)
            run_lengths.append(current_len)
            current_value = v
            current_len = 1

    # last run
    run_values.append(current_value)
    run_lengths.append(current_len)

    df_runs = pd.DataFrame({"value": run_values, "run_length": run_lengths})
    summary = (
        df_runs.groupby(["value", "run_length"])
        .size()
        .reset_index(name="count")
        .sort_values(["value", "run_length"])
    )
    return summary

In [ ]:

#  4. L2 sanity checks under V1
# 
# We now:
# - Compute class distributions for V1 train/val/test.
# - Inspect run-length distributions for L2 in each segment (head only).

# 
# These will help us understand class balance and persistence before training
# the E5–E6 models later.

v1_overview_rows = []

for split in ["train", "val", "test"]:
    df_split, X_split, y_split = build_xy_v1(df, split)
    n_rows = df_split.shape[0]
    n_label_na = df_split["y_L2"].isna().sum()
    n_feat_na = df_split[F_CORE_COLS].isna().any(axis=1).sum()

    v1_overview_rows.append(
        {
            "split": split,
            "n_rows": n_rows,
            "n_label_na": n_label_na,
            "n_rows_with_any_feature_na": n_feat_na,
        }
    )

    print(f"\n=== V1 {split.upper()} – L2 label distribution ===")
    dist = summarize_label_distribution(y_split, label_name="y_L2")
    print(dist)

    runs = run_length_distribution(df_split["y_L2"])
    print(f"\nRun-length summary (head) for V1 {split.upper()}:")
    print(runs.head())

v1_overview = pd.DataFrame(v1_overview_rows)
print("\n=== V1 NA / row overview ===")
print(v1_overview)

In [ ]:
# L2 sanity checks under V2 (folds F1–F4)

v2_overview_rows = []

for fold_id in sorted(v2_masks.keys()):
    for split in ["train", "test"]:
        df_split, X_split, y_split = build_xy_v2_fold(df, fold_id, split)
        n_rows = df_split.shape[0]
        n_label_na = df_split["y_L2"].isna().sum()
        n_feat_na = df_split[F_CORE_COLS].isna().any(axis=1).sum()

        v2_overview_rows.append(
            {
                "fold": fold_id,
                "split": split,
                "n_rows": n_rows,
                "n_label_na": n_label_na,
                "n_rows_with_any_feature_na": n_feat_na,
            }
        )

        print(f"\n=== V2 {fold_id} {split.upper()} – L2 label distribution ===")
        dist = summarize_label_distribution(y_split, label_name="y_L2")
        print(dist)

        runs = run_length_distribution(df_split["y_L2"])
        print(f"\nRun-length summary (head) for V2 {fold_id} {split.upper()}:")
        print(runs.head())

v2_overview = pd.DataFrame(v2_overview_rows)
print("\n=== V2 NA / row overview (all folds) ===")
print(v2_overview)


In [ ]:
# Simple bar plots for class proportions

def plot_label_distribution(dist_df, title):
    """
    dist_df: DataFrame with index = class labels, columns = ['count', 'proportion']
    """
    tmp = dist_df.reset_index().rename(columns={"index": "label"})
    fig = px.bar(
        tmp,
        x="label",
        y="proportion",
        text="proportion",
        title=title,
    )
    fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
    fig.update_layout(yaxis=dict(tickformat=".2f"))
    fig.show()


df_v1test, X_v1test, y_v1test = build_xy_v1(df, "test")
dist_v1test = summarize_label_distribution(y_v1test, label_name="y_L2")
print("\nNumeric summary – V1 TEST L2 label distribution:")
print(dist_v1test)
